# exp078 compact surface longtail gate train audit

## Contents

1. Setup and configuration
2. Input prediction alignment
3. Compact surface long-tail gate audit
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json

import pandas as pd

from settings import ExperimentPaths, load_config
from exp063_full_replay_reproducibility_guard import run_compact_surface_longtail_gate

paths = ExperimentPaths()
config = load_config()
paths.artifacts_dir.mkdir(parents=True, exist_ok=True)

summary_config = {
    "experiment": config["experiment"]["name"],
    "route": config["experiment"]["route"],
    "status": config["experiment"]["status"],
    "parent": config["lineage"]["parent"],
    "compact_parent": config["lineage"]["compact_parent"],
    "audit": config["audit"],
}
print(json.dumps(summary_config, indent=2))

## 2. Input prediction alignment

In [ ]:
base_predictions_path = config["data"].get("exp073_oof_predictions_local")
compact_predictions_path = config["data"].get("exp075_oof_predictions_local")

print("base:", base_predictions_path)
print("compact:", compact_predictions_path)
print("metric context:")
for item in config["lineage"].get("metric_context", []):
    print("-", item)

## 3. Compact surface long-tail gate audit

In [ ]:
audit = config["audit"]
summary = run_compact_surface_longtail_gate(
    output_dir=paths.artifacts_dir,
    base_predictions_path=base_predictions_path,
    compact_predictions_path=compact_predictions_path,
    mode_name=audit.get("selected_mode", "gpu_repro_guard_dp_threads8"),
    model_name=audit.get("selected_model", "lgb_mean"),
    weights=audit.get("weights", [0.05, 0.10, 0.20]),
    max_allowed_well_rmse_regression=audit.get("discussion_metric_guard", {}).get(
        "max_allowed_well_rmse_regression",
        0.25,
    ),
)
summary

## 4. Metrics and artifacts

In [ ]:
metrics_path = paths.artifacts_dir / "exp078_compact_surface_longtail_gate_metrics.csv"
bucket_path = paths.artifacts_dir / "exp078_compact_surface_longtail_gate_bucket_metrics.csv"
well_path = paths.artifacts_dir / "exp078_compact_surface_longtail_gate_well_metrics.csv"

metrics = pd.read_csv(metrics_path)
buckets = pd.read_csv(bucket_path)
wells = pd.read_csv(well_path)

print("Best policies")
display(metrics.head(12))

print("Tail buckets for best policy")
best_policy = metrics.iloc[0]["policy"]
display(buckets[buckets["policy"] == best_policy].sort_values(["segment_type", "segment"]))

print("Worst well regressions for best policy")
display(
    wells[wells["policy"] == best_policy]
    .sort_values("delta_rmse_vs_base", ascending=False)
    .head(20)
)

print("Artifacts")
for name in summary["artifacts"].values():
    print("-", paths.artifacts_dir / name)